# Using BERT to Vectorize Text

In [ ]:
from transformers import BertTokenizer, BertModel                     
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import glob
import os

## Read in the Articles

In [ ]:
# https://archive.ics.uci.edu/dataset/311/sentence+classification 
input_folder = "../data/labeled_articles/"
file_list=glob.glob("../data/labeled_articles/*.txt")

dfs = []
for filename in file_list:
    
    df = pd.read_csv(filename, delimiter='\t', names=['label', 'text'], index_col=None, header=None, on_bad_lines='skip')
    df['file'] = os.path.basename(filename)
    if df['text'].isnull().any():
        df['text']=df["label"].str[5:]  # Fix for files with missing text values
        df['label']=df["label"].str[:4]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

## Initialize the Transformer Model

In [ ]:
bert_model = BertModel.from_pretrained('bert-base-uncased')           
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')   
 

In [ ]:
def batch_embed_text(bert_model, tokenizer, text_iterable, batch_size=256):
    ''' This helper method will batch embed an iterable 
        of text using a given tokenizer and bert model '''
    encoding = tokenizer.batch_encode_plus(text_iterable, padding=True)
    input_ids = np.vstack(encoding['input_ids'])
    attention_mask = np.vstack(encoding['attention_mask'])
    
    def batch_array_idx(np_array, batch_size):
        for i in tqdm(range(0, np_array.shape[0], batch_size)):
            yield i, i + batch_size
            
    embedded = None
 
    for start_idx, end_idx in batch_array_idx(
        input_ids, batch_size=batch_size):
        batch_bert = bert_model(
            torch.tensor(input_ids[start_idx:end_idx]),
            attention_mask=torch.tensor(attention_mask[start_idx:end_idx])
        )[1].detach().numpy()
        if embedded is None:
            embedded = batch_bert
        else:
            embedded = np.vstack([embedded, batch_bert])
 
    return embedded 

In [ ]:
bert_data = batch_embed_text(bert_model, bert_tokenizer, df['text'], batch_size=128)


In [ ]:
pd.DataFrame(bert_data)
pd.DataFrame(bert_data).to_csv("../data/bert_article_vectors.csv", index=False) 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
print(bert_data[0].shape)
print(bert_data[0].reshape(1, -1).shape)
similarity_score = cosine_similarity(bert_data[0].reshape(1, -1), bert_data[1].reshape(1, -1))
print("Similarity Score:", similarity_score)